Creates a very basic de novo simulated dataset, with a more realistic ratio of true positive to true negative than `simple_spread` provides.

This is similar to `simple_de_novo_simulation.ipynb`

# Imports

In [1]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np
import dask.dataframe as dd
 

%load_ext autoreload
%autoreload 2

2025-09-12 18:16:42.251570: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-12 18:16:42.255675: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/Code-Server/4.17.0/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

# Create cluster

In [2]:
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(memory_limit='24GB')
client=Client(cluster)

# Seting experimiental design parameters

In [3]:
#first, we define the new parameters we want to assign to this object.

new_cell_number=pd.Series({"reference":1000,"blood":2000,"neuron":1000})

#we make up 3x replicates
new_zi=pd.Series({"replicate_A":0.02,"replicate_B":0.021})

new_min=1
new_max=200

new_MOI=60

In [4]:
#next, let's create a bounds object from these parameters and the bounds of the shendure data.
artificial_bounds=scm.SHENDURE_BOUNDS.copy(
    min_mpra_umi=new_min,
    max_mpra_umi=new_max,
    zi=new_zi,
    cells_per_cell_type=new_cell_number)
    
artificial_bounds.set_effective_moi(new_MOI)

# Creating an artificial library

`cat shendure_counts_grouped.txt | cut -f4,11 | grep "^minP" | cut -f2 | awk '{ sum += $1; n++ } END { if (n > 0) print sum / n }'`
Produces `0.0727759` as the average MPRA UMIs per cell for minP.

In [7]:
#making up the CREs
spread_gt,spread_hypothesis=scm.activity_spread(
    cell_types=list(new_cell_number.keys()),
    minimum=new_min,
    maximum=new_max,
    minp_value=0.0727759,
    total=1000,
    frac_active=0.5,
    ct_specificity=.2)

library=scm.simulate_library(CREs=spread_gt["cre_id"],
                 library_model=artificial_bounds.library_model)

scMPRAforge: INFO: 83.6% of active elements are not cell-type specific.


# Performing de-novo simulation

In [8]:
batch_plus=scm.de_novo_simulation(
                        simulation_replicates=3,
                        experiment_bounds=artificial_bounds,
                        ground_truth=spread_gt,
                        library=library)

In [9]:
batch_plus.gamut(client)

# Testing parameter recapitulation

In [10]:
test_data=batch_plus.simulated_scMPRA[0].result()

In [11]:
test_data.ortho_filter()

scMPRAforge: INFO: Dropped 126 of 2988 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [12]:
primordial=scm.ortho()
primordial.criss_cross(client=client,
                       dat=test_data)
primordial.extract_params(client)

In [13]:
primordial.compute_model_qc()

regression error on inactive_268


In [14]:
cell=[]
for i in primordial.by_cell_qc:
    working=primordial.by_cell_qc[i]["dat"]
    working["cell_type"]=i
    cell.append(working)

cell=pd.concat(cell).reset_index()
cell.merge(spread_gt,on=["cre_id","cell_type"])

,cre_id,mu,mean(umis_mpra_bc),cell_type,true_mean
0,active_0,60.722160,58.436681,blood,75.286722
1,active_1,41.149135,39.677966,blood,37.551817
2,active_10,106.513454,102.519582,blood,94.017972
3,active_100,68.185934,65.445652,blood,62.020999
4,active_101,15.309075,14.752044,blood,16.628219
...,...,...,...,...,...
2857,inactive_96,0.147987,0.142857,reference,0.072776
2858,inactive_97,0.075646,0.073034,reference,0.072776
2859,inactive_98,0.084906,0.081967,reference,0.072776
2860,inactive_99,0.072400,0.069892,reference,0.072776


In [15]:
cell=[]
for i in primordial.by_cre_qc:
    working=primordial.by_cre_qc[i]["dat"]
    working["cre_id"]=i
    cell.append(working)

cell=pd.concat(cell).reset_index()
cell.merge(spread_gt,on=["cre_id","cell_type"])

,cell_type,mu,mean(umis_mpra_bc),cre_id,true_mean
0,blood,60.628515,58.436681,active_0,75.286722
1,neuron,70.028780,67.620968,active_0,75.286722
2,reference,79.890787,77.063492,active_0,75.286722
3,blood,40.095153,39.677966,active_1,37.551817
4,neuron,151.961331,148.900000,active_1,134.739033
...,...,...,...,...,...
2857,neuron,0.184022,0.114754,inactive_75,0.072776
2858,reference,0.096933,0.056604,inactive_75,0.072776
2859,reference,NaN,0.064935,inactive_269,0.072776
2860,reference,NaN,0.181818,inactive_59,0.072776


# Save

In [16]:
data_root="/gpfs/gibbs/pi/reilly/tabula_data"
batch_plus.save(data_root,"simulated/activity_de_novo")
spread_hypothesis.to_tsv(f"{data_root}/simulated/activity_de_novo_hypotheses.tsv")

In [17]:
cluster.close()

2025-09-12 20:04:10,099 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:32773' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'_extract_theta-c4c10ce578269706cc58272397868aae', '_extract_zi-a66cd711717dfa2983ef8e0b564fa1b3', '_label_tensorzinb_regressors-07469c9a3bb2eba049b24fdce825ed9c', '_extract_zi-b4544105a655ffb80ab23c0925a19cb1', '_smart_matrix-a4d9f834601c614fff38f0ee99a6bf74', '_extract_zi-a0e46e99edb547fb788d7605efb631d6', '_extract_mu-2d85c7b2dbc2d411d27557c34dca23ba', '_extract_mu-e8ea39bdfc66180626f62d479ee9a7d1', '_smart_matrix-1a945fad6ae956a24f328d79c6f7026d', '_extract_theta-066d7b2b9cbc15ee62d3ca02ea33a2da', '_label_tensorzinb_regressors-56248f1b7b70b9829a480acd2bd6cbd9', '_smart_matrix-fc0a2a842a5b9ca6a6395d7840324889', '_smart_matrix-01b29c4b336bb5043ffd825497261faa', '_extract_theta-c265db77fa2c4a25109f72f246ebd909', '_smart_matrix-d15da3602d18a56486bbf9537630d13b', '_extract_mu-212026077ff494389be70591

2025-09-12 20:04:14,094 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-09-12 20:04:14,095 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-09-12 20:04:14,096 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-09-12 20:04:14,110 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
